# Tutorial 1 - Wireless Networking



In [15]:
# Some imports
import math
pi = math.pi

### Question 3


There is growing interest in connecting devices using a network of small satellites.
For instance, SpaceX’s Starlink network provides such a service. Most existing networks
primarily focus on providing downlink connectivity from satellites to ground devices. Now,
consider the possibility of IoT devices communicating directly with an overhead satellite. The
objective is to enable the transmission of small amounts of information between the IoT device
and the satellite. Assume the satellite is in a low Earth orbit (LEO), approximately 400
kilometers above the Earth’s surface.

#### a) What frequency would you choose for communication? What factors would you consider when selecting an appropriate frequency for this application?


> 400 km is some serious distance. I'll consider a frequency that can travel for so far without too much interference. I'll also consider an antenna size that can fit onto the IoT device. Given that, medium to low freq? 3MHz?

> *model answer:*

> Lower freq has advantages for IoT devices constrained by power limitations, as they enable extended battery life. It allows signals to propagate further at the same transmit power. 868/900 MHz ISM bands can be considered, 100-200MHz bands also possible w regulatory approval

#### b) What kind of antenna would you consider for the IOT device? How does the form of device influence the choice of antenna? What can you do mitigate challenges of antenna selection due to the form of the IoT device?

> I'll consider a directional antenna pointing upwards. IoT devices are often small so I'll need an antenna that fits into the device. 

> Model Answer:

> Challenges: devices such as air tags, wearables, may not be able to accommodate highly directional antennas such as parabolic antennas, In these cases, a smaller lower-gain omnidirectional antenna may be the only feasible option. 

> To compensate for lower gain, satellite may need to transmit a much stronger signal, or the device may only be able to establish communication where there's minimal obstruction between satellite and device

#### c) Given that the satellite’s signal sensitivity is -120 dBm, and the IoT device transmits at 30 dBm with a transmit antenna effective area $\pi * (0.05)^2$, what antenna gain would be required at the satellite for the chosen frequency in Part (a)?

In [35]:
# Here we have a couple of values:

# On the satellite side:
Pr_dbm = -120 #dBm Gr = Power Receiver

# On the transmitter side:
Pt_dbm = 30 #dBm Pt = Power Transmitter
Ae = pi*0.05**2

# General params
f = 900e6 # assuming 900Mhz
c = 3e8
wl = c/f #wl = wavelength

# What antenna gain would be required at the satellite for 900Mhz? 
# Gt = ?

Gt_linear = (4*pi*Ae)/wl**2
print("Gt_linear: ", Gt_linear)

Gt_linear:  0.8882643960980423


> So at this point, I've calculated the antenna gain on the IoT device, i.e. the Gt = 0.888, gain of transmitter. But first, I'll need to convert it to dBi:

In [36]:
Gt_dbi = 10*math.log10(Gt_linear)
print("Gt_dbi: ", Gt_dbi, " dBi")

Gt_dbi:  -0.514577451724074  dBi


> Model answer:

> The gain of transmitter is -0.514dBi, a negative gain, meaning it's less efficient. This is often possible when antenna on the device has been miniaturized. 

> At this point, I'll need to use the FSPL equation to calculate the FSPL first:

$$FSPL = 20\log_{10}(d) + 20\log_{10}(f) + 20\log_{10}(\frac{4\pi}{c}))$$ 

In [39]:
log10 = math.log10
d = 400000
FSPL_db = 20*log10(d) + 20*log10(f) + 20*log10(4*pi/c)
print("FSPL: ", FSPL_db, " dB")


FSPL:  143.56782220139445  dB


> This tells us how much signal will attenuate across the distance. 

> So now, using the other form of FSPL equation: 
$$P_r^{dBm} = P_t^{dBm} + G_t^{dBi} + G_r^{dBi} - FSPL^{dB} $$

> ...we can find G_r

$$G_r^{dBi} = P_r^{dBm} - P_t^{dBm} - G_t^{dBi} + FSPL^{dB} $$

In [40]:
# Quick print of our known values:
print("Pr_dbm: ", Pr_dbm)
print("Pt_dbm: ", Pt_dbm)
print("Gt_dbi: ", Gt_dbi)
print("FSPL_db: ", FSPL_db)

Pr_dbm:  -120
Pt_dbm:  30
Gt_dbi:  -0.514577451724074
FSPL_db:  143.56782220139445


In [42]:
Gr_dbi = Pr_dbm - Pt_dbm - Gt_dbi + FSPL_db
print("Gr_dbi: ", Gr_dbi)

Gr_dbi:  -5.917600346881471


> So, at satellite, any antenna with gain greater than -5.91 dBi would work! - validated answer

#### Digression into another form of Friss Equation

> Now, let's try to see how the friss equation would be different if we were do do multiplication. In linear scale, the friss equation would be like so:

$$P_r^{linear} = G_r^{linear}*G_t^{linear}*(\frac{c}{4\pi fd})^n*P_t^{linear}$$

> For context, the following would be an example question:

We have a receiver with effective radius of 100cm receiving signals at 2GHz from a transmitter that transmits at a power of 100W and gain of 40dB (or 10,000). Assume path loss exponent is 2. 

1. What is the gain of receiver antenna?
2. What is the received power if the receiver is 1km away from the transmitter?
3. If the receiver is receiving signals at 900 MHz frequency (instead of 2GHz), what is a) and b) again?

In [43]:
# In this case, we have the following:
r = 1 #m
f = 2e9
Pt_linear = 100 # Watts
Gt_linear = 10000 # + 10000 watts over 100 
# Gr_linear = ?
Ae = pi
wl = 3e8/2e9
Gr_linear = 4*pi*pi/wl**2
print("Gr_linear: ", Gr_linear)

Gr_linear:  1754.5963379714415


In [44]:
Gr_dbi = 10*log10(Gr_linear)
print("Gr_dbi: ", Gr_dbi)

Gr_dbi:  32.441772186048674


> 2. What is the received power if the receiver is 1km away from the transmitter?

$$P_r^{linear} = G_r^{linear}*G_t^{linear}*(\frac{c}{4\pi fd})^n*P_t^{linear}$$

In [46]:
# Pr_linear = ? 
Pr_linear = Gr_linear * Gt_linear * Pt_linear * (c/(4*pi*f*1000))**2
print("Pr_linear: ", Pr_linear, " watts")

Pr_linear:  0.25000000000000006  watts


#### d) If a frequency of 13.56 MHz is used for communication:

- What would be the link budget and antenna gain required? Assume IoT device has
an antenna gain of 6 dBi.


> So in this case, you need the link budget of the transmission. Let's recap: you're sending a 30 dBm signal from a transmitter at 13.56MHz, with gain of 6dBi. The FSPL is 143 dB, and you'll need the receiver sensititity to be -120dBm. The qn is asking: what's the link budget, and what would be the satellite antenna gain? The antenna gain I've found previously is -5.91 dBi for f of 900MHz. I'll need to find the antenna gain for 13.56MHz. 

In [50]:
# Here we have a couple of values:

# On the satellite side:
Pr_dbm = -120 #dBm Gr = Power Receiver

# On the transmitter side:
Pt_dbm = 30 #dBm Pt = Power Transmitter
Ae = pi*0.05**2

# General params
f = 13.56e6 # assuming 900Mhz
c = 3e8
wl = c/f #wl = wavelength

# What antenna gain would be required at the satellite for 13.56Mhz? 
# Gt = ?

# Gt_linear = (4*pi*Ae)/wl**2
# print("Gt_linear: ", Gt_linear)
Gt_dbi = 6
print("Gt_dbi: ", Gt_dbi, " dBi")

Gt_dbi:  6  dBi


> Here we see that the antenna gain at transmitter is -36dBi, which is super low? Are my calculations correct?

> I'll need to also calculate Link Budget

$$P_r^{dBm} = P_t^{dBm} + G_t^{dBi} + G_r^{dBi} - L_{total}^{dB}$$

In [51]:
log10 = math.log10
d = 400000
FSPL_db = 20*log10(d) + 20*log10(f) + 20*log10(4*pi/c)
print("FSPL: ", FSPL_db, " dB")

FSPL:  107.12816580322882  dB


In [56]:
# -120 = 30 + 6 + Gr_dBi - 107.12
Gr_dbi = -120 - 30- 6+107.12
print("Gr_dbi: ", Gr_dbi)

Gr_dbi:  -48.879999999999995


> The gain required on satellite is -48.88 dBi. 

Link Budget:

In [57]:
Pr_dbm = Pt_dbm + Gt_dbi + Gr_dbi - FSPL_db
print("Pr_dbm: ", Pr_dbm)

Pr_dbm:  -120.00816580322882


#### Why is this frequency unsuitable for satellite communication with IoT devices?

In [ ]:
# # wl of 13Mhz
# wl = c/13.56e6
# print("wl: ", wl, "m")

wl:  22.123893805309734 m


> A dipole at 13.56MHz is 22/2 ~11m, too large for IoT. Furthermore, HF signals are reflected by the ionosphere, preventing line of sight communication with satellites. 3) Low freq like 13.56Mhz support minimal bandwidth, making them unsuitable for modern IoT apps that require much higher data rate. 